# 04 — Outer Loop: LLM-Guided Structure Search

Structure Descent outer loop:
1. **Diagnostics** — compute validation metrics and residual analysis
2. **LLM proposal** — send structured report to Claude; receive JSON proposal
3. **Validate** — parse and grammar-check the proposed structure S'
4. **Inner loop** — fit weights on S', compute Score(S', θ')
5. **Accept/reject** — keep S' only if Score(S', θ') > Score(S, θ)

This is Metropolis-Hastings over model structures, with the LLM as the proposal distribution.

**Requires**: `ANTHROPIC_API_KEY` in `.env`

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import json
from pathlib import Path
from dotenv import load_dotenv

from src.dsl import DSLStructure
from src.inner_loop import run_inner_loop
from src.outer_loop import generate_diagnostics, prompt_llm, apply_proposal, structure_descent_step
from src.evaluation import compute_metrics, compute_residuals, print_metrics

load_dotenv('../.env')

DATA_DIR = Path('../data')

In [ ]:
# Load state from inner loop notebook
with open(DATA_DIR / 'train_features.pkl', 'rb') as f:
    feat_data = pickle.load(f)
with open(DATA_DIR / 'current_state.pkl', 'rb') as f:
    state = pickle.load(f)

features_list  = feat_data['features_list']
chosen_indices = feat_data['chosen_indices']
customer_ids   = feat_data['customer_ids']
categories     = feat_data['categories']

with open(DATA_DIR / 'train_choices.pkl', 'rb') as f:
    train_choices = pickle.load(f)
metadata = [e['metadata'] for e in train_choices[:len(features_list)]]

current_structure = DSLStructure.from_dict(state['structure'])
current_score     = state['score']
metrics           = state['metrics']
residuals         = state['residuals']

print(f'Starting: {current_structure}')
print(f'Score: {current_score:.2f}')

## Step 1: Inspect the diagnostic report

This is exactly what the LLM will receive as its proposal prompt.

In [ ]:
report = generate_diagnostics(current_structure, current_score, metrics, residuals)
print(report)

## Step 2: Define the fit function

Wraps the inner loop so the outer loop can call `fit_fn(structure)`.

In [ ]:
def fit_fn(structure: DSLStructure):
    """Run inner loop and return (weights, posterior_score)."""
    # NOTE: in practice you should re-extract features for the new structure.
    # Here we reuse existing features for terms that overlap.
    # A full implementation would call extract_features() for the new term set.
    return run_inner_loop(
        structure, features_list, chosen_indices, customer_ids, categories,
        sigma=10.0, tau=1.0, nu=0.5
    )

print('fit_fn defined.')

## Step 3: Single outer-loop iteration (manual inspection)

In [ ]:
new_structure, new_score, accepted = structure_descent_step(
    current_structure=current_structure,
    current_score=current_score,
    metrics=metrics,
    residuals=residuals,
    fit_fn=fit_fn,
    iteration=1,
)

if accepted:
    print(f'\n✓ New structure accepted: {new_structure}')
    current_structure = new_structure
    current_score = new_score
else:
    print(f'\n✗ Proposal rejected. Keeping: {current_structure}')

## Step 4: Full structure descent run (N iterations)

In [ ]:
N_ITERATIONS = 8  # Increase for full experiment

history = [{
    'iteration': 0,
    'structure': str(current_structure),
    'score': current_score,
    'accepted': True,
}]

for i in range(1, N_ITERATIONS + 1):
    # Refresh metrics and residuals for current structure
    weights, _ = fit_fn(current_structure)
    metrics = compute_metrics(features_list, chosen_indices,
                              weights.get_weights, customer_ids, categories)
    residuals = compute_residuals(features_list, chosen_indices,
                                  weights.get_weights, customer_ids, categories, metadata)
    
    new_structure, new_score, accepted = structure_descent_step(
        current_structure, current_score, metrics, residuals, fit_fn, iteration=i
    )
    
    if accepted:
        current_structure = new_structure
        current_score = new_score
    
    history.append({
        'iteration': i,
        'structure': str(current_structure),
        'score': current_score,
        'accepted': accepted,
    })

print(f'\nFinal structure: {current_structure}')

## Search history

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
display(hist_df)

plt.figure(figsize=(8, 3))
plt.plot(hist_df['iteration'], hist_df['score'], marker='o')
plt.scatter(hist_df[hist_df['accepted']]['iteration'],
            hist_df[hist_df['accepted']]['score'], color='green', zorder=5, label='Accepted')
plt.xlabel('Outer loop iteration')
plt.ylabel('Posterior score')
plt.title('Structure Descent: posterior score over iterations')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save final structure
with open(DATA_DIR / 'final_structure.pkl', 'wb') as f:
    pickle.dump({'structure': current_structure.to_dict(),
                 'score': current_score,
                 'history': history}, f)
print(f'Saved. Final: {current_structure}  |  Score: {current_score:.2f}')